<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/Mx_Oscar_Hdez_de_Hands_On_Integraci%C3%B3n_entre_Llama_y_WhatsApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "VERIFY_TOKEN"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_TOKEN')
PHONE_ID = userdata.get('PHONE_ID')
WABA_ID = userdata.get('WABA_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 135 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (519 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../z

In [ ]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 8.2 MB/s eta 0:00:00
  

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import json

app = FastAPI()

mensajes_recibidos = []
ids_procesados = set()

In [ ]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [ ]:
# Definir la funcion real y su descripcion para Llama: consultar disponibilidad

import re
from datetime import datetime

DISPONIBILIDAD = {
    "2026-10-06": ["09:00", "11:00", "15:00"],
    "2026-10-07": ["10:00", "13:00"],
}

MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}

def normalizar_fecha(fecha_texto):
    fecha_texto = fecha_texto.strip().lower()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha_texto):
        _, mes, dia = fecha_texto.split("-")
        return f"{datetime.now().year:04d}-{mes}-{dia}"

    match = re.search(r"(\d{1,2})\s+de\s+([a-zñ]+)(?:\s+de\s+(\d{4}))?", fecha_texto)
    if match:
        dia, mes_texto, anio_texto = match.groups()
        mes = MESES.get(mes_texto)
        if mes:
            anio = int(anio_texto) if anio_texto else datetime.now().year
            return f"{anio:04d}-{mes:02d}-{int(dia):02d}"

    return fecha_texto

def consultar_disponibilidad(fecha=None):
    if not fecha:
        dias = ", ".join(DISPONIBILIDAD.keys())
        return f"Los dias con horarios disponibles son: {dias}. Dime cual te interesa para darte los horarios exactos."

    fecha = normalizar_fecha(fecha)
    horarios = DISPONIBILIDAD.get(fecha, [])
    if not horarios:
        return f"No hay horarios disponibles para el {fecha}."
    return f"Horarios disponibles para el {fecha}: {', '.join(horarios)}."

HERRAMIENTA_DISPONIBILIDAD = {
    "type": "function",
    "name": "consultar_disponibilidad",
    "description": "Consulta los horarios disponibles para llevar el auto al taller. Si el usuario da una fecha especifica, se le devuelven los horarios de ese dia; si pregunta de forma general (sin fecha), se le devuelve la lista de dias con disponibilidad.",
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {
                "type": "string",
                "description": "Fecha a consultar, en formato YYYY-MM-DD. Si el usuario no menciona el año, asume que es el año actual. Omite este parametro si el usuario no dio una fecha especifica."
            }
        },
        "required": []
    }
}

In [ ]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

import unicodedata

def quitar_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

PREGUNTAS_FRECUENTES = {
    "horario": "El taller abre de lunes a sabado, de 8:00 a 18:00.",
    "garantia": "Todas las reparaciones tienen 90 dias de garantia en mano de obra.",   # raiz, no palabra completa
    "pago": "Aceptamos efectivo, tarjeta y transferencia.",
    "cotizacion": "La cotizacion es gratuita, solo se cobra si decides hacer la reparacion.",  # raiz
}

def buscar_pregunta_frecuente(pregunta):
    pregunta_norm = quitar_acentos(pregunta.lower())
    for raiz, respuesta in PREGUNTAS_FRECUENTES.items():
        if raiz in pregunta_norm:
            return respuesta
    return "No tenemos una respuesta guardada para esa pregunta; un miembro del equipo puede confirmarte directamente."

HERRAMIENTA_FAQ = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca la respuesta a una pregunta frecuente del taller (horarios, garantia, formas de pago, cotizaciones) en una base de preguntas frecuentes real.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "La pregunta del usuario, en texto libre."
            }
        },
        "required": ["pregunta"]
    }
}

HERRAMIENTAS = [HERRAMIENTA_DISPONIBILIDAD, HERRAMIENTA_FAQ]
FUNCIONES_DISPONIBLES = {
    "consultar_disponibilidad": consultar_disponibilidad,
    "buscar_pregunta_frecuente": buscar_pregunta_frecuente,
}

#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria y function calling

SYSTEM_PROMPT = (
    "Eres el asistente de un taller mecanico. "
    "Responde de forma breve (2-3 lineas). "
    "Solo usa una herramienta cuando el usuario pregunte algo especifico que la herramienta pueda responder "
    "(disponibilidad de una fecha, o una pregunta frecuente concreta). No uses ninguna herramienta para "
    "confirmaciones o respuestas cortas del usuario (como 'si, por favor' o una hora). "
    "No puedes agendar, confirmar ni programar citas de ninguna forma — solo puedes consultar disponibilidad. "
    "Si el usuario quiere agendar una cita, dile que un miembro del equipo se pondra en contacto para confirmarla. "
    "Si una herramienta no encuentra una respuesta, dilo honestamente en vez de inventar un dato. "
    "Si una herramienta SI regresa un resultado real, siempre comunicalo claramente al usuario; nunca lo ignores ni te niegues a compartirlo. "
    "Responde siempre de forma directa con la informacion final, basandote unicamente en el resultado real de las "
    "herramientas; nunca inventes datos que no vengan de una herramienta. "
    "Nunca describas tu proceso, ni menciones nombres de funciones, parametros o claves de un resultado."
)

def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
    )

    llamadas_funcion = [item for item in respuesta.output if item.type == "function_call"]
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
            funcion = FUNCIONES_DISPONIBLES[llamada.name]
            argumentos = json.loads(llamada.arguments)
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
        )

    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

In [ ]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        wamid = mensaje["id"]
        if wamid in ids_procesados:
            return {"status": "ok"}  # Meta reenvio el mismo mensaje; ya se proceso, se ignora
        ids_procesados.add(wamid)
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass
    return {"status": "ok"}

In [ ]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [ ]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama

import requests
from llama_stack_client import APITimeoutError, APIConnectionError, APIStatusError

FRASES_URGENCIA = ["me quede varado", "no arranca", "choque", "tuve un accidente", "emergencia"]
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "asistente humano"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "que tal", "buen dia", "buen día"]

def es_solo_saludo(texto_low):
    texto_limpio = texto_low.strip(" ,.!¡¿?")
    return texto_limpio in FRASES_SALUDO

def enviar_aviso_de_error(numero):
    try:
        enviar_respuesta(numero, "Estamos teniendo un problema tecnico. Intenta de nuevo en un momento.")
    except Exception:
        pass  # si ni el aviso de error se pudo mandar, no hay mas que intentar aqui

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(numero, "Eso suena urgente: comunicate directamente a nuestra linea de asistencia en carretera.")
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            elif es_solo_saludo(texto_low):
                enviar_respuesta(numero, "Hola, soy el asistente del taller mecanico. ¿En que puedo ayudarte?")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)

        except APITimeoutError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_timeout",
                "detalle": "Llama Stack conecto pero la generacion tardo demasiado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except APIConnectionError as e:
            llama_stack_vivo = proceso_llama_stack.poll() is None
            ollama_vivo = proceso_ollama.poll() is None
            if not llama_stack_vivo:
                causa, detalle = "llama_stack_caido", f"El proceso de Llama Stack ya no esta corriendo (Ollama vivo: {ollama_vivo})."
            elif not ollama_vivo:
                causa, detalle = "ollama_caido", "El proceso de Ollama ya no esta corriendo."
            else:
                causa, detalle = "conexion_desconocida", "No se pudo conectar a Llama Stack aunque ambos procesos parecen vivos."
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": causa, "detalle": detalle, "error": repr(e)})
            enviar_aviso_de_error(numero)

        except APIStatusError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_stack_error_http",
                "detalle": f"Llama Stack respondio con error HTTP {e.status_code}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except KeyError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "herramienta_no_reconocida",
                "detalle": f"El modelo pidio ejecutar una herramienta que no existe: {e}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except requests.exceptions.RequestException as e:
            # Este es el unico que NO intenta avisar al usuario: si fallo mandar el mensaje
            # a WhatsApp, intentar mandar el aviso de error fallaria por la misma razon.
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "whatsapp_api_error",
                "detalle": "Fallo el envio a la API de WhatsApp (graph.facebook.com).",
                "error": repr(e),
            })

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "desconocido",
                "detalle": "Error no clasificado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [ ]:
# Escenario 1: Mandar mensajes al bot por disponibilidad general

# Mensaje 1:
# Buenas tardes.

# Mensaje 2:
# Necesito llevar mi carro al taller, ¿qué días tienen espacio?

# Mensaje 3:
# Perfecto, ¿y a qué hora el 6 de octubre?

In [ ]:
# Escenario 2: Mandar mensajes al bot por tema de urgencia

# Mensaje 1:
# Hola. Necesito ayuda.
# Acabo de chocar y mi carro no arranca.

# Mensaje 2:
# Comunícame con un asistente humano.

In [ ]:
# Escenario 3: Mandar mensajes al bot por pago y cotización

# Mensaje 1:
# Buenas, ¿cuánto cuesta una cotización?

# Mensaje 2:
# ¿Y qué métodos de pago aceptan?

# Mensaje 3:
# Perfecto, entonces agéndame para el 7 de octubre a las 10.

In [ ]:
# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

# Mensaje 1:
# Hola, ¿tienen servicio de grúa?

# Mensaje 2:
# ¿También atienden motos, o solo carros?

In [ ]:
# Visualizar resultados

print(f'Conversaciones: {conversaciones_por_numero}\n')
print(f'Mensajes recibidos: {mensajes_recibidos}\n')
print(f'Respuestas enviadas: {respuestas_enviadas}\n')
print(f'Errores procesados: {errores_procesamiento}\n')

Conversaciones: {}

Mensajes recibidos: []

Respuestas enviadas: []

Errores procesados: []



In [ ]:
# Verificar almacenamiento en Llama Stack

numero_a_revisar = '525536699933'

if numero_a_revisar not in conversaciones_por_numero:
    print("No hay mensajes almacenados en estos momentos.")
else:
    conv_id = conversaciones_por_numero[numero_a_revisar]
    items = client.conversations.items.list(conversation_id=conv_id)

    for item in items.data:
        if item.type == "message":
            print(item.role, "->", item.content)
        elif item.type == "function_call":
            print("function_call ->", item.name, item.arguments)
        elif item.type == "function_call_output":
            print("function_call_output ->", item.output)
        else:
            print(item.type, "->", item)

assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='No tengo información para decirte cuando será, afortunadamente el taller abre de lunes a sábado de 8:00 a 18:00.', annotations=[], logprobs=[], type='output_text')]
function_call -> buscar_pregunta_frecuente {"pregunta":"No me han llamado"}
function_call_output -> No tenemos una respuesta guardada para esa pregunta; un miembro del equipo puede confirmarte directamente.
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='No me han llamado', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Un 

### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [ ]:
# Reiniciar el almacenamiento en Llama Stack

proceso_llama_stack.terminate()
import time; time.sleep(3)

!rm -rf ~/.llama/distributions/starter

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [ ]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores

mensajes_recibidos = []
respuestas_enviadas = []
errores_procesamiento = []
conversaciones_por_numero = {}

In [ ]:
# Finalizar proceso de tunelizacion

ngrok.kill()
print('Proceso de ngrok cerrado.')

Proceso de ngrok cerrado.


In [ ]:
# "Configuración de la API" de Meta, normalmente arriba del Phone Number ID

WABA_ID = "1389496259965043"

In [ ]:
# 1. Ver qué apps están suscritas a este WABA

r = requests.get(
    f"https://graph.facebook.com/v26.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)
print("Suscripciones actuales:", r.json())

Suscripciones actuales: {'data': [{'whatsapp_business_api_data': {'category': 'Herramientas de software', 'link': 'https://www.facebook.com/games/?app_id=1055942273865487', 'name': 'Mx.Oscar.Hdez', 'id': '1055942273865487'}}]}


In [ ]:
# 2. Suscribir esta app al WABA

r2 = requests.post(
    f"https://graph.facebook.com/v26.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)
print("Resultado de suscripción:", r2.json())

Resultado de suscripción: {'success': True}
